In [1]:
import numpy as np
import pandas as pd

initial_df = pd.read_excel("data_clear.xlsx", parse_dates=["event_time"])

## Обрабатываем изначальный DataFrame, чтобы работать только с заданным промежутком времени

In [23]:
# 24.09.2020 - 28.02.2021
def set_time_intervals(start: str, end: str, df: pd.DataFrame) -> pd.DataFrame:
    start = list(map(int, start.split(".")))
    end = list(map(int, end.split(".")))
    start_ts = pd.Timestamp(day=start[0], month=start[1], year=start[2], tz="UTC")
    end_ts = pd.Timestamp(day=end[0], month=end[1], year=end[2], tz="UTC") + pd.Timedelta(days=1)
    time_mask = (start_ts <= df["event_time"]) & (end_ts > df["event_time"]) 
    return df[time_mask]


df = set_time_intervals("24.09.2020", "28.02.2021", initial_df)
df

,event_time,event_type,product_id,category_code,brand,price,user_id,user_session
0,2020-09-24 11:57:06+00:00,view,1996170,electronics.telephone,NaN,2775.30,1515915625519379968,LJuJVLEjPT
1,2020-09-24 11:57:26+00:00,view,139905,computers.components.cooler,zalman,1492.92,1515915625519379968,tdicluNnRY
2,2020-09-24 11:57:27+00:00,view,215454,NaN,NaN,853.47,1515915625513230080,4TMArHtXQy
3,2020-09-24 11:57:33+00:00,view,635807,computers.peripherals.printer,pantum,9901.47,1515915625519010048,aGFYrNgC08
4,2020-09-24 11:57:36+00:00,view,3658723,NaN,cameronsino,1380.69,1515915625510739968,aa4mmk0kwQ
...,...,...,...,...,...,...,...,...
884959,2021-02-28 23:55:01+00:00,view,953226,NaN,NaN,19134.78,1515915625611020032,FRLqIttxKU
884960,2021-02-28 23:58:05+00:00,view,1715907,electronics.video.tv,starwind,6962.61,1515915625611020032,g6WqPf50Ma
884961,2021-02-28 23:58:09+00:00,view,4170534,electronics.clocks,amazfit,5648.04,1515915625611020032,xNIJBqZdkd
884962,2021-02-28 23:58:14+00:00,view,888273,electronics.telephone,NaN,883.92,1515915625611020032,9pCbKMIcSx


## 1). Анализ пользователей, процентное соотношение всех событий к количеству сессий

In [3]:
from typing import Optional

class EventTypeAnalysis:
    def __init__(self, df: pd.DataFrame):
        self.LABEL = "Анализ совершённых событий пользователями"
        self.df: Optional[pd.DataFrame] = None
        self.total_sessions = len(df)

        self.views = df["has_view"].sum()
        self.carts = df["has_cart"].sum()
        self.purchases = df["has_purchase"].sum()

        self.view_percentage = round(self.views / self.total_sessions, 3)
        self.cart_percentage = round(self.carts / self.total_sessions, 3)
        self.purchases_percentage = round(self.purchases / self.total_sessions, 3)

    def make_analysis(self) -> pd.DataFrame:
        self.df = pd.DataFrame(
            columns=[self.LABEL],
            index=[
                "Колличество сессий",
                "Просмотры",
                "Добавлений в карзину",
                "Покупок",
                "% просмотров",
                "% добавлений в карзину",
                "% покупок"
                ])
        self.df.loc["Колличество сессий", self.LABEL] = self.total_sessions
        self.df.loc["Просмотры", self.LABEL] = self.views
        self.df.loc["Добавлений в карзину", self.LABEL] = self.carts
        self.df.loc["Покупок", self.LABEL] = self.purchases
        self.df.loc["% просмотров", self.LABEL] = self.view_percentage
        self.df.loc["% добавлений в карзину", self.LABEL] = self.cart_percentage
        self.df.loc["% покупок", self.LABEL] = self.purchases_percentage
        return self.df

event_type_df = df.groupby(["user_session"])["event_type"].apply(set).reset_index()
event_type_df["has_view"] = event_type_df["event_type"].apply(lambda x: "view" in x)
event_type_df["has_cart"] = event_type_df["event_type"].apply(lambda x: "cart" in x)
event_type_df["has_purchase"] = event_type_df["event_type"].apply(lambda x: "purchase" in x)

event_type_analysis = EventTypeAnalysis(event_type_df)
event_type_analysis.make_analysis()
# event_type_df[event_type_df["has_view"] == False]


,Анализ совершённых событий пользователями
Колличество сессий,16990
Просмотры,16935
Добавлений в карзину,1119
Покупок,667
% просмотров,0.997
% добавлений в карзину,0.066
% покупок,0.039


## 2). Анализ времени между событиями, сколько пользователь тратит времени между просмотром товара, добавлением в корзину и покупкой

In [ ]:
# не доделал

session_time_sorted_df = df.sort_values(["user_session", "event_time"])
session_time_sorted_df

,event_time,event_type,product_id,category_code,brand,price,user_id,user_session
27192,2020-09-30 17:52:02+00:00,view,1675002,electronics.tablet,xiaomi,36646.14,1515915625509230080,000c34fa-991f-442a-8e07-8c472269bec6
9364,2020-09-26 18:59:06+00:00,view,3582066,computers.components.tv_tuner,d-color,2431.65,1515915625520059904,002DmERG1w
8955,2020-09-26 16:55:42+00:00,view,1247635,electronics.telephone,sirius,3645.30,1515915625519680000,009so5Mgnj
2481,2020-09-25 03:44:48+00:00,view,1756713,electronics.telephone,NaN,1822.65,1515915625519579904,00scna0OEj
2487,2020-09-25 03:46:21+00:00,view,1429384,electronics.telephone,NaN,696.00,1515915625519579904,00scna0OEj
...,...,...,...,...,...,...,...,...
22989,2020-09-29 19:55:01+00:00,view,275243,NaN,tp-link,709.92,1515915625520950016,zzrmGmDaIC
1130,2020-09-24 16:29:43+00:00,view,957592,electronics.tablet,samsung,1223.22,1515915625519480064,zzt9kKgXen
25849,2020-09-30 12:31:41+00:00,view,893196,computers.components.videocards,sapphire,18626.70,1515915625521110016,zztRHEozV0
25852,2020-09-30 12:32:12+00:00,view,877060,computers.components.videocards,msi,16294.23,1515915625521110016,zztRHEozV0


## 3). Аномалии в ценах, посмотреть бывают ли покупки одного товара по очень разной цене

In [24]:
purchases_df = df[df["event_type"] == "purchase"]
purchases_df

,event_time,event_type,product_id,category_code,brand,price,user_id,user_session
45,2020-09-24 12:04:10+00:00,purchase,1507291,computers.components.power_supply,supermicro,18928.59,1515915625519389952,xn6SHCnZtk
82,2020-09-24 12:15:06+00:00,purchase,822426,computers.peripherals.camera,logitech,10731.45,1515915625513570048,2gngxS29Ts
100,2020-09-24 12:19:01+00:00,purchase,4060928,electronics.video.tv,NaN,7762.14,1515915625518129920,3yFCkx2KKW
132,2020-09-24 12:25:18+00:00,purchase,4060928,electronics.video.tv,NaN,7762.14,1515915625518129920,3yFCkx2KKW
150,2020-09-24 12:29:49+00:00,purchase,1283197,computers.peripherals.nas,zyxel,10769.73,1515915625519350016,3jFpdbozOd
...,...,...,...,...,...,...,...,...
884905,2021-02-28 23:16:45+00:00,purchase,4154620,computers.components.videocards,msi,57126.81,1515915625596740096,h4fcX0qpOc
884913,2021-02-28 23:20:48+00:00,purchase,500058,computers.peripherals.printer,pantum,5829.00,1515915625610970112,CxMKMQDRAN
884917,2021-02-28 23:23:11+00:00,purchase,500058,computers.peripherals.printer,pantum,5829.00,1515915625610970112,CxMKMQDRAN
884925,2021-02-28 23:26:07+00:00,purchase,500058,computers.peripherals.printer,pantum,5829.00,1515915625610970112,CxMKMQDRAN


In [36]:

product_prices_df = df[df["event_type"] == "purchase"].groupby("product_id")["price"].apply(set).reset_index()
product_prices_df["price_count"] = product_prices_df["price"].apply(lambda x: len(x))
product_prices_df[product_prices_df["price_count"] > 1]
# если таблица пустая, значит все товары были куплены по одной цене

,product_id,price,price_count


In [ ]:
# purchase_rating_df = purchases_df.groupby("category_code")["product_id"].count().sort_values(ascending=False).reset_index()
# purchase_rating_df.to_csv("output.csv")

## 4). Анализ корзин

In [61]:

carts_df = purchases_df.groupby("user_session")["category_code"].apply(set).reset_index()
carts_df["cart_length"] = carts_df["category_code"].apply(len)
carts_df.sort_values("cart_length", ascending=False).head(10)

,user_session,category_code,cart_length
10527,QdBpqEeQGH,"{computers.components.videocards, computers.co...",9
13549,YO3XxnSFXX,"{computers.components.power_supply, computers....",8
22068,uDu86hUZL1,"{computers.components.videocards, computers.co...",8
24114,zMr79OiVmW,"{computers.components.power_supply, computers....",7
23293,xM3yZfQYhj,"{computers.components.videocards, computers.co...",7
8297,KqZdoypn7P,"{computers.components.videocards, computers.co...",7
4120,AIUQQQVmVd,"{computers.components.power_supply, computers....",7
19902,oXsQs6aJAr,"{computers.components.videocards, computers.co...",7
11324,SlNpDATCvK,"{computers.components.videocards, computers.co...",7
6045,F6ohHpTTBU,"{computers.components.power_supply, computers....",7
